# 06. 1D-CNN with Derived BS-Side CSI Temporal Features (h=5, 300-User)

Trains a **Derived BS-Side CSI 1D-CNN** (history $h=5$, $L=6$ timestep window).

### Key design decisions:
- **100% Base-Station-side features:** `rss`, `sinr`, `aoa_*`, temporal derivatives — measured inside the gNodeB PHY layer. No UE private data.
- **15% single-antenna UE ratio** (3GPP Rel-15/16 realistic mix).
- **Multi-task heads:** position + speed + uncertainty, trained jointly.
- **Early stopping** (patience=8 epochs, min delta 0.05 m).
- Uses shared utilities from `utils/csi_dataset.py` and `utils/training_utils.py`.

## Imports & Environment Setup

In [ ]:
import sys, os, time, warnings, copy, datetime
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists(): break
    if PROJECT_ROOT.parent == PROJECT_ROOT: break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))

from pipelines.multi_user_pipeline_regression import _read_bs_position_3d
from pipelines.multi_user_200_pipeline import make_unseen_user_split
from utils.environment_viz import plot_environment_layout, analyze_distance_and_interference_errors
from utils.csi_dataset import DerivedCSI1DDataset, load_and_prepare_data
from utils.training_utils import (
    EarlyStopping, collect_test_predictions, print_per_antenna_benchmark,
    apply_kalman_smoother, plot_trajectories
)

RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
OUT_DIR  = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'multi_user_poc' / f'cnn_h5_{RUN_TIMESTAMP}'
PLOT_DIR = OUT_DIR / 'trajectory_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')

## Global Hyperparameters

In [ ]:
SEED             = 42
TRAIN_USER_RATIO = 0.80
H_TARGET         = 5          # history depth -> L = 6 timesteps
SEQUENCE_LENGTH  = H_TARGET + 1
SINGLE_ANT_RATIO = 0.15

BATCH_SIZE       = 512
EPOCHS           = 60         # generous budget; early stopping will cut this
LEARNING_RATE    = 3e-4
WEIGHT_DECAY     = 1e-2
DROPOUT_RATE     = 0.35

# Multi-task loss weights
LAMBDA_POS       = 1.0
LAMBDA_SPEED     = 0.2
LAMBDA_UNCERTAINTY = 0.05

# Early stopping
ES_PATIENCE      = 8          # stop after 8 epochs with no improvement
ES_MIN_DELTA     = 0.05       # minimum improvement (metres) to reset counter

# Kalman post-processing
PROCESS_NOISE_STD = 0.5
R_STD             = 15.0

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | Epochs: {EPOCHS} | EarlyStopping: patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}m')

## Load & Prepare Multi-User Dataset

In [ ]:
data_base = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25'
dirs = sorted(list(data_base.glob('sim_data_300users_*')))
if not dirs: dirs = sorted(list(data_base.glob('sim_data_200users_*')))
if not dirs: raise FileNotFoundError('No sim_data_300users_* or sim_data_200users_* dataset found!')
DATA_DIR = dirs[-1]
print(f'Dataset: {DATA_DIR}')

bs_pos = np.array(_read_bs_position_3d(DATA_DIR))

# load_and_prepare_data: loads raw data, applies AoA noise model,
# downsamples single-antenna UEs to 15%, computes 13 derived CSI features
df_filt, multi_uids, keep_single = load_and_prepare_data(
    DATA_DIR, bs_pos, seed=SEED, single_ant_ratio=SINGLE_ANT_RATIO
)
print(f'Filtered: {len(df_filt):,} records | {len(multi_uids)} Multi-Ant + {len(keep_single)} Single-Ant [15%]')
print(f'Total users: {df_filt["user_id"].nunique()} | Steps: {df_filt["step_index"].nunique()} unique')

## 2D Spatial Environment Map

In [ ]:
env_map_path = OUT_DIR / 'environment_spatial_layout.png'
interferers  = [[-60, 53, 10], [53, -60, 10]]
fig, ax = plot_environment_layout(bs_pos, interferers=interferers,
                                  scenario_label='3GPP 38.901 Mixed UMi',
                                  save_path=env_map_path)
plt.show()
print(f'[SAVED] {env_map_path}')

## Build Dataset (13 Channels, h=5, `[B, 13, 6]`)

In [ ]:
df_split, train_users, test_users = make_unseen_user_split(df_filt, train_ratio=TRAIN_USER_RATIO, seed=SEED)
df_train = df_split[df_split['split']=='train'].reset_index(drop=True)
df_test  = df_split[df_split['split']=='test'].reset_index(drop=True)

train_ds = DerivedCSI1DDataset(df_train, h=H_TARGET)
test_ds  = DerivedCSI1DDataset(df_test,  h=H_TARGET,
                               sig_mean=train_ds.sig_mean,   sig_std=train_ds.sig_std,
                               stat_mean=train_ds.stat_mean, stat_std=train_ds.stat_std,
                               targ_mean=train_ds.targ_mean, targ_std=train_ds.targ_std,
                               speed_mean=train_ds.speed_mean, speed_std=train_ds.speed_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f'Train: {len(train_ds):,} sequences ({len(train_users)} users) | Test: {len(test_ds):,} sequences ({len(test_users)} users)')
print(f'Input shape: [B, C=13, L={SEQUENCE_LENGTH}]')

## 1D-CNN Architecture (3 Conv Blocks, Multi-Task Heads)

In [ ]:
class DerivedCSI1DCNNNet(nn.Module):
    def __init__(self, in_channels=13, static_dim=3, dropout=0.35):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels, 32,  kernel_size=3, padding=1), nn.BatchNorm1d(32),  nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Conv1d(32,          64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Conv1d(64,          128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool1d(1)
        )
        fusion_dim = 128 + static_dim
        self.shared_trunk = nn.Sequential(
            nn.Linear(fusion_dim, 128), nn.LeakyReLU(0.1), nn.BatchNorm1d(128), nn.Dropout(dropout)
        )
        self.head_pos   = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.1), nn.Linear(64, 2))
        self.head_speed = nn.Sequential(nn.Linear(128, 32), nn.LeakyReLU(0.1), nn.Linear(32, 1))
        self.head_unc   = nn.Sequential(nn.Linear(128, 32), nn.LeakyReLU(0.1), nn.Linear(32, 1), nn.Softplus())

    def forward(self, x_seq, x_static):
        cnn_feat = self.conv_block(x_seq).squeeze(-1)
        fused    = torch.cat([cnn_feat, x_static], dim=1)
        feat     = self.shared_trunk(fused)
        return self.head_pos(feat), self.head_speed(feat), self.head_unc(feat)


model = DerivedCSI1DCNNNet(in_channels=13, static_dim=3, dropout=DROPOUT_RATE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## Train 1D-CNN with Early Stopping

In [ ]:
criterion_pos   = nn.SmoothL1Loss()
criterion_speed = nn.SmoothL1Loss()
criterion_unc   = nn.SmoothL1Loss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# EarlyStopping from utils/training_utils.py
early_stopping = EarlyStopping(patience=ES_PATIENCE, min_delta=ES_MIN_DELTA, verbose=True)

print(f'Training 1D-CNN for up to {EPOCHS} epochs (early stopping: patience={ES_PATIENCE})...')
print('='*110)
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train()
    tr_loss, tr_errs = 0.0, []
    for batch in train_loader:
        seq, stat, t_pos, t_spd = batch['seq'].to(device), batch['static'].to(device), batch['target'].to(device), batch['speed'].to(device)
        optimizer.zero_grad()
        p_pos, p_spd, p_unc = model(seq, stat)
        norm_errs = torch.norm(p_pos.detach() - t_pos, dim=1, keepdim=True)
        loss = LAMBDA_POS*criterion_pos(p_pos,t_pos) + LAMBDA_SPEED*criterion_speed(p_spd,t_spd) + LAMBDA_UNCERTAINTY*criterion_unc(p_unc,norm_errs)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tr_loss += loss.item() * len(t_pos)
        p_m = p_pos.detach().cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        t_m = t_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        tr_errs.extend(np.linalg.norm(p_m - t_m, axis=1))

    current_lr = scheduler.get_last_lr()[0]; scheduler.step()
    tr_loss /= len(train_ds); tr_mae = np.mean(tr_errs)

    model.eval(); val_loss, val_errs = 0.0, []
    with torch.no_grad():
        for batch in test_loader:
            seq, stat, t_pos, t_spd = batch['seq'].to(device), batch['static'].to(device), batch['target'].to(device), batch['speed'].to(device)
            p_pos, p_spd, p_unc = model(seq, stat)
            norm_errs = torch.norm(p_pos - t_pos, dim=1, keepdim=True)
            loss = LAMBDA_POS*criterion_pos(p_pos,t_pos) + LAMBDA_SPEED*criterion_speed(p_spd,t_spd) + LAMBDA_UNCERTAINTY*criterion_unc(p_unc,norm_errs)
            val_loss += loss.item() * len(t_pos)
            pred_m = p_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
            targ_m = t_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
            val_errs.extend(np.linalg.norm(pred_m - targ_m, axis=1))

    val_loss /= len(test_ds); val_mae = np.mean(val_errs)
    ep_time = time.time() - t_ep
    t_elapsed = time.time() - t_start
    eta_min = (t_elapsed / epoch) * (EPOCHS - epoch) / 60.0

    is_best_str = ' [BEST *]' if val_mae < early_stopping.best_val_mae else ''
    should_stop = early_stopping(val_mae, model, epoch, save_path=OUT_DIR / 'best_cnn_model.pt')
    if epoch % 2 == 0 or epoch == 1 or epoch == EPOCHS or is_best_str:
        print(f'Epoch {epoch:2d}/{EPOCHS} | Train MAE: {tr_mae:6.3f}m | Val MAE: {val_mae:6.3f}m | LR: {current_lr:.2e} | {ep_time:4.1f}s | ETA: {eta_min:4.1f}min{is_best_str}')
    if should_stop: break

print('='*110)
total_time = time.time() - t_start
early_stopping.restore_best(model)
print(f'Restored best model from epoch {early_stopping.best_epoch} | Best Val MAE: {early_stopping.best_val_mae:.3f}m')
print(f'Total training time: {total_time:.1f}s ({total_time/60:.1f} min) | Stopped at epoch: {epoch}/{EPOCHS}')
best_epoch = early_stopping.best_epoch

## Evaluate Best Model — Per-Antenna & Spatial Breakdown

In [ ]:
results = collect_test_predictions(model, test_loader, device, train_ds)
preds_raw   = results['preds_raw']
targs       = results['targs']
preds_speed = results['preds_speed']
preds_unc   = results['preds_unc']
uids        = results['uids']
dts         = results['dts']
n_ants      = results['n_ants']

errs_2d = np.linalg.norm(preds_raw - targs, axis=1)
mae_overall, p50, p90 = print_per_antenna_benchmark(
    errs_2d, n_ants, uids,
    model_name='1D-CNN (NB06)', h_target=H_TARGET, best_epoch=best_epoch
)

interferers = [[-60, 53, 10], [53, -60, 10]]
analyze_distance_and_interference_errors(preds_raw, targs, bs_pos, interferers=interferers, report_dir=OUT_DIR)

## Kinematic Post-Processing: KF & RTS Smoother

In [ ]:
kf_preds, rts_preds = apply_kalman_smoother(
    preds_raw, targs, uids, dts, test_users,
    process_noise_std=PROCESS_NOISE_STD, R_std=R_STD
)

errs_rts  = np.linalg.norm(rts_preds - targs, axis=1)
mae_rts   = np.mean(errs_rts)
red_rts   = (mae_overall - mae_rts) / mae_overall * 100

print(f'Raw 1D-CNN MAE:          {mae_overall:.3f}m')
print(f'RTS Smoother MAE:        {mae_rts:.3f}m  ({red_rts:+.1f}%)')
print('-'*60)
print('Per-antenna RTS improvement:')
for nant in [4, 2, 1]:
    mask = (n_ants == nant)
    if not np.any(mask): continue
    r_raw = np.mean(errs_2d[mask]); r_rts = np.mean(errs_rts[mask])
    red   = (r_raw - r_rts) / r_raw * 100
    print(f'  [{nant}-Antenna] Raw: {r_raw:.3f}m -> RTS: {r_rts:.3f}m ({red:+.1f}%)')

## Trajectory Tracking Plots — 5 Unseen Test Users

In [ ]:
plot_trajectories(
    targs, preds_raw, rts_preds, uids, n_ants, test_users,
    plot_dir=PLOT_DIR,
    scatter_color='dodgerblue',
    model_label='1D-CNN'
)